# NB05a Quantification

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/CSBiology/BIO-BTE-06-L-7/gh-pages?filepath=NB05a_Quantification.ipynb)

[Download Notebook](https://github.com/CSBiology/BIO-BTE-06-L-7/releases/download/NB05a/NB05a_Quantification.ipynb)

1. Quantification Theory
    1. Targeted quantification
    2. (i) Targeted acquisition at peptide
    3. (ii) Targeted data analysis at peptide ion level
2. References

## Quantification Theory

To estimate the amount of individual proteins in complex mixtures, all peptide signals corresponding to a common protein serve as a 
proxy for their abundance. Peptide information needs to be obtained from multidimensional signal data detected by the mass spectrometer. 
All signals generated from one peptide ion species, often referred to as peptide feature, need to be grouped to form a three-dimensional peak 
along the m/z, ion intensity, and retention time dimension. This process is generally defined as peak detection or feature detection. 
Peak detection algorithms are a set of rules defining how neighboring signal points are joined. Whether noise filtering is done before or after 
peak detection strongly depends on the peak detection algorithm. Traditional approaches mainly focused on signal amplitude neglecting 
characteristic peak shapes as a common feature of chromatographic or spectroscopic peaks. These algorithms are prone to miss detection of low 
intensity peaks with a signal strength close to the noise level. To overcome these issues, techniques like smoothing, shape-matching and curve 
fitting are often implemented and applied. At the time, the most promising approach to do shape-matching and noise reduction in one step uses the 
continuous wavelet transformation (CWT).

In general, a CWT based approach describes a family of time-frequency-transformations often used in data compression and feature detection. 
The term is coined by the use of a wavelet, as a basis function which is “compared” to the signal. The point of highest correlation between the 
basis function and the signal reflects the location of the peak present. Due to the fact that MS derived peaks often follow the shape of a 
gaussian distribution, the *Mexican Hat* wavelet as the negative normalized second derivative of the Gaussian distribution is perfectly 
suited to find the peptide feature.

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/Wavelets.png)

**Figure 5: Schematic representation of the ‘Haar’-wavelet (blue) and the ‘Mexican Hat’- wavelet (green). **
The ‘Haar’-wavelet is named after its discoverer Alfred Haar and represents the first wavelet ever to be described. The ‘Mexican Hat’- or ‘Ricker’-wavelet is 
frequently used in the fields of signal detection and compression.

Depending on the quantification approach, the peptide features used for protein quantification might differ. In case of isotopic labeling, 
quantification means pairing features with the proper mass shift according to the utilized label. It is essential to account for the frequency 
of label incorporation when calculating the mass shift for the utilized label. Taking the ICAT method as an example, by which a heavy/light 
difference of 9 Dalton per cysteine is incorporated, the total mass shift is 9 Dalton times the number of cysteine within the sequence. 
Consequently, pairing peptide features for 15N labeling is even more challenging, as the mass shift is less discrete. Using stable 
isotope labeling, different peptide feature pairs belonging to the same protein can be treated as technical replicates and averaged to gain 
protein quantification. In contrast, the sum of all extracted peptide signals results in a label-free protein quantiﬁcation. Spectral counting 
computes abundance values from the number of times a peptide was successfully identiﬁed by tandem mass spectrometry (MS/MS) and combines all 
these events per protein. The spectral counting values can be normalized by the number of peptides theoretically expected from the particular 
protein. 

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/ComputationalProteinQuantification.png)

**Figure 6: Computational strategy of peptide and protein quantiﬁcation on based on stable isotope labeling or by label-free quantiﬁcation.**
(A) Label-free methods compare corresponding peptide abundances over different MS runs. The abundance is either 
estimated by the elution proﬁle les of the pep de ions or (B) in case of spectral counting, by the number of times a peptide was 
successfully identiﬁed (MS2). In contrast, methods based on differential stable isotope labeling analyze peptides pairs detected by 
their characteristic mass diﬀerence Δm/z. The abundance is estimated by the ratio of their corresponding elution proﬁles (C). Isobaric 
tagging methods (D) compare the reporter ion abundances in the fragmentation spectrum.
### Targeted quantification

Targeted proteomics has gained significant popularity in mass spectrometry‐based protein quantification as a method to detect proteins of 
interest with high sensitivity, quantitative accuracy and reproducibility. The two major strategies of (i) targeted acquisition at peptide, 
and (ii) targeted data analysis at peptide ion level need to be distinguished.
###(i) Targeted acquisition at peptide

In multiple reaction monitoring (MRM or SRM for single/selected reaction monitoring) simply predefined transitions are recorded. 
Knowledge about the targeted transitions from precursor to their corresponding fragment ions are needed and predefined in the mass 
spectrometer. MRM can be performed rapidly and is highly specific even for low abundant peptide ions in complex mixtures, but with the 
drawback of a necessary bias in the sense that only predefined peptides are measured.
### (ii) Targeted data analysis at peptide ion level

Data‐independent acquisition at the peptide level makes it possible to acquire peptide data for virtually all peptide ions present in a sample. 
In this strategy, a high‐resolution mass analyzer—such as an orbitrap or a time‐of‐flight—is used to constantly sample the full mass range 
at the peptide level during the entire chromatographic gradient. In a subsequent step, precursor ion chromatograms can be extracted by targeted 
data analysis. Those extracted-ion chromatogram (XIC) can be obtained to calculate the area under the curve and used for peptide quantification.

Let’s start and extract a XIC…


In [1]:
#r "nuget: FSharp.Stats, 0.4.3"
#r "nuget: BioFSharp, 2.0.0-beta5"
#r "nuget: BioFSharp.IO, 2.0.0-beta5"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: System.Data.SQLite, 1.0.113.7"
#r "nuget: BioFSharp.Mz, 0.1.5-beta"
#r "nuget: MzIO, 0.1.1"
#r "nuget: MzIO.SQL, 0.1.4"
#r "nuget: MzIO.Processing, 0.1.2"
#r "nuget: MzLite, 1.1.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"

open Plotly.NET
open FSharp.Stats
open BioFSharp
open System.IO
open System.Data.SQLite
open MzIO
open MzIO.Processing
open MzLite
open MzIO.MzSQL


Installed Packages BioFSharp, 2.0.0-beta5 BioFSharp.IO, 2.0.0-beta5 BioFSharp.Mz, 0.1.5-beta FSharp.Stats, 0.4.3 MzIO, 0.1.1 MzIO.Processing, 0.1.2 MzIO.SQL, 0.1.4 MzLite, 1.1.0 Plotly.NET, 4.2.0 Plotly.NET.Interactive, 4.2.0 System.Data.SQLite, 1.0.113.7

Loading extensions from `/home/paulinehans/.nuget/packages/plotly.net.interactive/4.2.0/lib/netstandard2.1/Plotly.NET.Interactive.dll`

We now want to extract the XIC for the peptide where we previously calculated the matching score.

Since we need several mass spectrometry scans to quantify over the retention time, we connect to our database 
and index the entries according to their retention time.


**We know from the MS2 measurement, that our peptide had its match at a retention of around 20 min**. We create a query 
to the database to extract the intensities of all peaks that are +/-5 min of our retention time and within 0.04 m/z of our peptide of interest. 
After we are done, we close the connection to the database.


With the data in the MzLite File we now create a XIC for every peptide

In [ ]:
// Code-Block 1id

let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory;"/home/paulinehans/Downloads/qPgr1_1_5 1.mzlite"|]

let runID = "sample=0"
let mzReader = new MzIO.MzSQL.MzSQL(path)
let peaksReader = new MzIO.MzSQL.MzSQL(path)
let cn = mzReader.Open()
let cn2 = peaksReader.Open()
let transaction = mzReader.BeginTransaction()


//Indexes all spectra of the related sample run
let idx = MzIO.Processing.Query.getMS1RTIdx mzReader runID
idx 
//easier format
let index = idx |> Seq.toArray
index

let getMs1 = mzReader.ReadMassSpectra runID
let extractData =
    getMs1
    |> Seq.choose (fun x ->
        match MzIO.Processing.MassSpectrum.getMsLevel x with
        | 1 ->
            Some {
                id = x.ID
                precursorMZ = MzIO.Processing.MassSpectrum.getPrecursorMZ x
                retentiontime = MzIO.Processing.MassSpectrum.getScanTime x
                mzIntensity = 
                    PeakArray.mzIntensityArrayOf (
                        peaksReader.ReadSpectrumPeaks x.ID
                    )
            }
        | _ -> None
    )

    |> Seq.toArray

let allInfo = 
    extractData 
    |> Array.map (fun x -> x.id, x.precursorMZ, x.retentiontime, x.mzIntensity)
allInfo

index value 0 (sample=1 period=1 cycle=2187 experiment=1, -1, 19.997416666667, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2187 experiment=1 Item2 -1 Item3 19.997416666667 Item4 (System.Double[], System.Double[]) Item1 [ 350.01939271783255, 350.0470726128987, 350.07034357515937, 350.084878226574, 350.13530974043977, 350.2331353558375, 350.3008463007071, 350.31591189283193, 350.33516106904887, 350.7761393004178, 350.79465555540355, 350.83338255447615, 350.85333663320915, 350.88213494122743, 350.8976853655609, 350.91177837593307, 350.93626884775955, 351.0004445944736, 351.04386092538925, 351.09070414697413 ... (5767 more) ] Item2 [ 29.559638357173412, 14.05782517453531, 2.3649438544453005, 31.401956371928463, 9.460680767534768, 16.558459156374283, 24.051666091992274, 4.731565269253224, 13.012170624313057, 11.442089396740585, 7.102180875637032, 60.10841932846347, 19.729983905295967, 26.04467612179917, 11.312555748594377, 21.44171548166014, 9.076846224672863, 22.49666022554493, 7.104710530330749, 34.73679576788129 ... (5767 more) ] 1 (sample=1 period=1 cycle=2186 experiment=1, -1, 19.992416666667, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2186 experiment=1 Item2 -1 Item3 19.992416666667 Item4 (System.Double[], System.Double[]) Item1 [ 350.0273846562378, 350.05110581892, 350.07630994204743, 350.1125323124189, 350.17486723813687, 350.2332468896462, 350.2462782588825, 350.2937486753636, 350.31087496249677, 350.34775052560985, 350.7677674377932, 350.7911962001995, 350.79896457927293, 350.8084502821185, 350.83533536356344, 350.86611017364686, 350.89399874276495, 350.9529837044844, 350.9864467643835, 351.00186780553076 ... (5740 more) ] Item2 [ 25.224432298929855, 4.335630842803084, 34.8176573392152, 18.78934119054918, 15.111738540034025, 20.369400786682945, 12.09055013320608, 9.72567586904347, 18.40048893199082, 9.069254721680409, 16.17653465812623, 6.049969225946938, 2.3674014522953257, 12.363284347924264, 78.91757120296461, 11.70663094563622, 83.66044385008138, 7.761534852360796, 19.733708005094286, 4.7361964329809325 ... (5740 more) ] 2 (sample=1 period=1 cycle=2185 experiment=1, -1, 19.987416666667, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2185 experiment=1 Item2 -1 Item3 19.987416666667 Item4 (System.Double[], System.Double[]) Item1 [ 350.00484265468754, 350.02478875310425, 350.0502403988529, 350.08241912673617, 350.1024799508557, 350.16012072750084, 350.2093627390465, 350.2621091848961, 350.3300790081299, 350.77813930089206, 350.8108341987799, 350.8378173499505, 350.8820157594256, 350.9016053958303, 350.91950519107365, 350.94372033187153, 350.9624575602957, 350.98318642619586, 350.99634227113285, 351.01734155641947 ... (5630 more) ] Item2 [ 4.335337935540906, 20.36356483157533, 10.773332474821018, 37.314174718079585, 20.234566757898165, 7.095760397152162, 12.089912542313641, 24.57592704407716, 12.091954017171247, 30.512286182542425, 21.701596656819788, 53.13821953169224, 23.150767102887926, 17.100470945254642, 27.098346497564307, 7.10369552094221, 6.051443624426156, 12.76099138013933, 7.104228079617769, 7.367554744280369 ... (5630 more) ] 3 (sample=1 period=1 cycle=2184 experiment=1, -1, 19.982316666667, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2184 experiment=1 Item2 -1 Item3 19.982316666667 Item4 (System.Double[], System.Double[]) Item1 [ 350.03334866150004, 350.05763391085355, 350.0768667257659, 350.11773439275385, 350.1377825775859, 350.2372484201362, 350.2596579181787, 350.2966288719538, 350.3296510824434, 350.3717114665189, 350.7637374860313, 350.7845629825217, 350.80564740792306, 350.83495894445093, 350.8574933263684, 350.8814464249648, 350.9061173258951, 350.9261441632415, 350.9488293855519, 350.9953057485045 ... (5769 more) ] Item2 [ 32.71286902155168, 11.036234079028873, 40.46753041556599, 14.84777724298283, 11.826089809011364, 38.76796658991566, 11.039382092087635, 14.1943167272359, 11.8291520213063, 4.731935963883416, 15.51

In [ ]:
//Code-Block2 updated

let retentiontime = 
    allInfo 
    |> Array.map (fun x -> 
        x
        |> fun (_,_,rt,_,_) -> rt )

let mzAtCharge2 = 
    allInfo 
    |> Array.map (fun x -> 
        x
        |> fun (_,precursorMZ,_,_,_) -> precursorMZ )

let rtQuery = 
    retentiontime
    |> Array.map (fun x -> MzIO.Processing.Query.createRangeQuery x 5.)
    

let mzQuery = 
    mzAtCharge2
    |> Array.map (fun x -> MzIO.Processing.Query.createRangeQuery x 0.04)

let zipQuerys =
    rtQuery
    |> Array.zip mzQuery

let xic = 
    zipQuerys
    |> Array.map (fun (x,y) -> MzIO.Processing.Query.getXIC mzReader idx x y )

let inData = 
    xic 
    |> Array.map (fun x -> x.)

let xicCharts = 
    xic 
    |> Array.map (fun x -> 
        x
        |> Chart.Point
        |> Chart.withXAxisStyle "Retention Time"
        |> Chart.withYAxisStyle "Intensity/Score"
        |> Chart.withSize (900.,900.)
        )






In [ ]:
// Code-Block 2

let retentionTime = 9.350283333333
let mzAtCharge2   = 909.4677415

let rtQuery = MzIO.Processing.Query.createRangeQuery retentionTime 5.

let mzQuery = MzIO.Processing.Query.createRangeQuery mzAtCharge2 0.04

let xic = 
    MzIO.Processing.Query.getXIC mzReader idx rtQuery mzQuery  
    |> Array.map (fun p -> p.Rt , p.Intensity)
    
transaction.Dispose()

let xicChart =
    xic
    |> Chart.Point
    |> Chart.withXAxisStyle "Retention Time"
    |> Chart.withYAxisStyle "Intensity/Score"
    |> Chart.withSize (900.,900.)

xicChart

We have now the XIC in our hands and can use the second derivative to identify peaks with our trace.


In [ ]:
// Code-Block 3

// get all peaks
let peaks = 
    xic
    |> Array.unzip
    |> (fun (ret, intensity) ->
        FSharp.Stats.Signal.PeakDetection.SecondDerivative.getPeaks 0.1 2 13 ret intensity
        )

peaks |> Array.head


{ Apex = { Index = 19\n XVal = 4.4477\n YVal = 35.57820533 }\n LeftLiftOff = Some { Index = 13\n XVal = 4.417566667\n YVal = 0.0 }\n LeftEnd = { Index = 12\n XVal = 4.412583333\n YVal = 0.0 }\n RightLiftOff ... Apex { Index = 19\n XVal = 4.4477\n YVal = 35.57820533 } Index 19 XVal 4.4477 YVal 35.578205326442 LeftLiftOff Some({ Index = 13\n XVal = 4.417566667\n YVal = 0.0 }) Value { Index = 13\n XVal = 4.417566667\n YVal = 0.0 } Index 13 XVal 4.417566666667 YVal 0 LeftEnd { Index = 12\n XVal = 4.412583333\n YVal = 0.0 } Index 12 XVal 4.412583333333 YVal 0 RightLiftOff Some({ Index = 25\n XVal = 4.4778\n YVal = 0.0 }) Value { Index = 25\n XVal = 4.4778\n YVal = 0.0 } Index 25 XVal 4.4778 YVal 0 RightEnd { Index = 25\n XVal = 4.4778\n YVal = 0.0 } Index 25 XVal 4.4778 YVal 0 LeftSidedConvolved False RightSidedConvolved True XData [ 4.412583333333, 4.417566666667, 4.422566666667, 4.42765, 4.432633333333, 4.437633333333, 4.4427, 4.4477, 4.4527, 4.457766666667, 4.462733333333, 4.467716666667, 4.4728, 4.4778 ] YData [ 0, 0, 0, 0, 0, 0, 0, 35.578205326442, 0, 0, 0, 0, 0, 0 ]

The peak model includes numerus information. Therefore we can mark the apices of the peaks we identified.


In [ ]:
// Code-Block 4

let apices =
    peaks
    |> Array.map (fun peak -> peak.Apex.XVal,peak.Apex.YVal)

let apicesChart=
    [    
        Chart.Point(apices, Name = "apices")
        |> Chart.withMarkerStyle(Size = 15)
        Chart.Point(xic, Name = "XIC")

    ]
    |> Chart.combine
    |> Chart.withXAxisStyle "Retention Time"
    |> Chart.withYAxisStyle "Intensity"
    |> Chart.withSize (900., 900.)

apicesChart


<!-- Plotly chart will be drawn inside this DIV -->

We can then go ahead and characterize our peak and quantify the area under the fitted curve.


In [ ]:
// Code-Block 5

// get peak at "ret=51.95" from all peaks "peaks"
let quantifiedXIC = 
    BioFSharp.Mz.Quantification.HULQ.getPeakBy peaks retentionTime
    // quantify peak of interest
    |> BioFSharp.Mz.Quantification.HULQ.quantifyPeak 
    
quantifiedXIC.Area


72.15802851910779

The peak model gives us all the information we need for our peptide of interest. If we want to see what we quantified, we can take an 
exponential modified gaussian function using the parameters given by the peak model and plot it together with the previously extracted XIC.


In [ ]:
// Code-Block 6

let eval x = 
    Fitting.NonLinearRegression.Table.emgModel.GetFunctionValue (vector quantifiedXIC.EstimatedParams) x
eval 


FSI_0033+it@5-3

In [ ]:
// Code-Block 7

let quantifiedArea =
    xic 
    |> Array.map (fun (rt,i) -> rt, eval rt)

let quantifiedAreaChart =
    [
        Chart.Point(xic, Name="XIC")
        Chart.SplineArea(quantifiedArea, Name = "quantified XIC")
    ]
    |> Chart.combine
    |> Chart.withXAxisStyle (TitleText = "Retention Time", MinMax = (51., 58.))
    |> Chart.withYAxisStyle "Intensity"
    |> Chart.withSize (900., 900.)

quantifiedAreaChart


Error: System.IndexOutOfRangeException: Index was outside the bounds of the array.
   at FSharp.Stats.Fitting.NonLinearRegression.Table.emgModel@438.Invoke(Vector`1 parameterVector, Double xValue) in C:\Users\bvenn\source\repos\FSharp.Stats\src\FSharp.Stats\Fitting\NonLinearRegression.fs:line 440
   at <StartupCode$FSI_0034>.$FSI_0034.main@()
   at System.RuntimeMethodHandle.InvokeMethod(Object target, Void** arguments, Signature sig, Boolean isConstructor)
   at System.Reflection.MethodBaseInvoker.InvokeWithNoArgs(Object obj, BindingFlags invokeAttr)

## Questions

1. How does the Chart created by Code-Block 2 change, when you change the value of 'retentionTime' to 53.95? What does this parameter specify?
2. How does the Chart created by Code-Block 2 change, when you change the value of the parameter 'offset' from 5 to 10 or 20?
3. How does the Chart created by Code-Block 4 change, when you change the value of the parameter snr from 0.1 to 2.1? 
What does this parameter specify, what does the abbreviation snr stand for?
4. How does the Chart created by Code-Block 7 change, when you change the value of the parameter retentionTime in CodeBlock 5 to 55.15?
5. Have a look at the peaks, how are the peaks shaped, are the shapes symmetric?
6. What does the term "peak tailing" imply. 
7. What factors determine peak shape? Think of explanations (e.g. biochemical-interactions, detection method) for different peak shapes. 
8. How many parameters does the model have (see quantifiedXIC.EstimatedParams, Code-Block 6), what does the abbreviation "EMG" stand for and 
how is this function different from a gaussian function?
9. How could the fit created by the change in Code-Block 5, 6 and 8 profit from baseline correction?
